In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
"""
Clarity Detection System (SemEval-2026 Task 6)

- DeBERTa-v3-base encoder
- Ensemble of 5 models
- Test-Time Augmentation with dropout
"""

import os, gc, zipfile, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from sklearn.metrics import f1_score
from safetensors.torch import load_file
from collections import Counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 Device: {device}")

# ==========================================================
# 1. LOAD DATA
# ==========================================================
print("\n📂 Loading dataset...")
dataset = load_dataset("ailsntua/QEvasion")
dataset = dataset.filter(lambda x: x["clarity_label"] != "")

train_data = dataset["train"]
labels = sorted(set(train_data["clarity_label"]))
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
num_labels = len(labels)

counts = Counter(train_data["clarity_label"])
class_weights = torch.tensor(
    [len(train_data)/(num_labels*counts[id2label[i]]) for i in range(num_labels)],
    dtype=torch.float32
)

print(f"✅ Training data: {len(train_data)} examples")
print(f"✅ Labels: {labels}")
print(f"✅ Class distribution: {dict(counts)}")

# ==========================================================
# 2. TOKENIZATION 
# ==========================================================
print("\n🔤 Setting up tokenizer...")
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 320

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    texts = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(batch["question"], batch["interview_answer"])
    ]
    tok = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    tok["labels"] = [label2id[l] for l in batch["clarity_label"]]
    return tok

print("🔄 Tokenizing training data...")
tok_train = train_data.map(tokenize, batched=True, batch_size=512, remove_columns=train_data.column_names)
tok_train.set_format("torch")
print(f"✅ Tokenized {len(tok_train)} examples")

# ==========================================================
# 3. MODEL 
# ==========================================================
class BalancedModel(nn.Module):
    def __init__(self, base, n, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base, use_safetensors=True)
        h = self.encoder.config.hidden_size
        
        self.classifier = nn.Sequential(
            nn.Linear(h, h//2),
            nn.LayerNorm(h//2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h//2, n)
        )

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(input_ids, attention_mask, return_dict=True)
        
        # Mean pooling
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        
        logits = self.classifier(pooled)
        
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits, 
                labels, 
                weight=class_weights.to(logits.device)
            )
            
        return {"loss": loss, "logits": logits}

# ==========================================================
# 4. TRAIN 5 MODELS
# ==========================================================
def train_model(seed, out_dir):
    torch.manual_seed(seed)
    np.random.seed(seed)
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"\n{'='*60}")
    print(f"🏋️  Training model with seed {seed}")
    print(f"{'='*60}")
    
    model = BalancedModel(MODEL_NAME, num_labels)
    
    args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=4e-5,
        per_device_train_batch_size=20,
        num_train_epochs=4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        fp16=False,
        gradient_accumulation_steps=1,
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        report_to=[],
        seed=seed,
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
    )
    
    trainer.train()
    trainer.save_model(out_dir)
    
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"✅ Model saved to {out_dir}")

# Train 5 models for better ensemble diversity
print("\n" + "="*60)
print("🎯 TRAINING 5-MODEL ENSEMBLE")
print("="*60)

SEEDS = [42, 123, 456, 789, 2024]
for i, seed in enumerate(SEEDS, 1):
    print(f"\n[{i}/5] Training model {i}...")
    train_model(seed, f"./final_model_{i}")

print("\n✅ All 5 models trained successfully!")

# ==========================================================
# 5. LOAD EVALUATION DATA
# ==========================================================
print("\n" + "="*60)
print("📊 LOADING EVALUATION DATA")
print("="*60)

EVAL_URL = "https://raw.githubusercontent.com/konstantinosftw/CLARITY-SemEval-2026/main/dataset/clarity_task_evaluation_dataset.csv"
eval_data = load_dataset("csv", data_files=EVAL_URL)["train"]

def tok_eval(batch):
    texts = [f"Question: {q}\nAnswer: {a}" for q, a in zip(batch["question"], batch["interview_answer"])]
    return tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding="max_length")

tok_eval = eval_data.map(tok_eval, batched=True, remove_columns=eval_data.column_names)
tok_eval.set_format("torch")

print(f"✅ Evaluation data: {len(tok_eval)} examples")

# ==========================================================
# 6. PREDICT WITH TTA (Test-Time Augmentation)
# ==========================================================
print("\n" + "="*60)
print("🔮 GENERATING PREDICTIONS WITH TTA")
print("="*60)

def predict_with_tta(model_dir, dataset):
    """Predict with Test-Time Augmentation using dropout"""
    model = BalancedModel(MODEL_NAME, num_labels).to(device)
    model.load_state_dict(load_file(f"{model_dir}/model.safetensors"))
    
    all_logits = []
    
    # Pass 1: Normal inference (dropout off)
    model.eval()
    batch_logits = []
    with torch.no_grad():
        for i in range(0, len(dataset), 40):
            batch = {
                k: dataset[i:i+40][k].to(device) 
                for k in ["input_ids", "attention_mask"]
            }
            logits = model(**batch)["logits"].cpu()
            batch_logits.append(logits)
    all_logits.append(torch.cat(batch_logits))
    
    # Pass 2: Inference with dropout enabled (slight augmentation)
    model.train()
    batch_logits = []
    with torch.no_grad():
        for i in range(0, len(dataset), 40):
            batch = {
                k: dataset[i:i+40][k].to(device) 
                for k in ["input_ids", "attention_mask"]
            }
            logits = model(**batch)["logits"].cpu()
            batch_logits.append(logits)
    all_logits.append(torch.cat(batch_logits))
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    # Average both passes
    return torch.stack(all_logits).mean(0)

# Predict with all 5 models using TTA
print("\n🔮 Running 5-model ensemble with TTA...")
all_logits = []
for i in range(1, 6):
    print(f"   [{i}/5] Model {i} (with TTA)...")
    logits = predict_with_tta(f"./final_model_{i}", tok_eval)
    all_logits.append(logits)
    print(f"      ✅ Shape: {logits.shape}")

# Average all model predictions
print("\n🔄 Averaging ensemble predictions...")
final_logits = sum(all_logits) / len(all_logits)
pred_ids = final_logits.argmax(dim=-1).tolist()
preds = [id2label[i] for i in pred_ids]

print(f"✅ Generated {len(preds)} predictions")

# ==========================================================
# 7. ANALYZE PREDICTIONS
# ==========================================================
print("\n" + "="*60)
print("📊 PREDICTION ANALYSIS")
print("="*60)

dist = Counter(preds)
print("\n📊 Final prediction distribution:")
for label in labels:
    count = dist.get(label, 0)
    pct = count/len(preds)*100
    print(f"   {label:20s}: {count:4d} ({pct:5.1f}%)")

# ==========================================================
# 8. SAVE SUBMISSION
# ==========================================================
print("\n" + "="*60)
print("💾 SAVING SUBMISSION")
print("="*60)

with open("prediction", "w") as f:
    for p in preds:
        f.write(str(p) + "\n")

print("✅ Saved predictions to 'prediction' file")

with zipfile.ZipFile("clarity_final_submission.zip", "w") as z:
    z.write("prediction")

print("✅ Created clarity_final_submission.zip")

# ==========================================================
# 9. SUMMARY
# ==========================================================
print("\n" + "="*60)
print("="*60)
print("\n📁 File: clarity_final_submission.zip")
print("   ✓ 5 models")
print("   ✓ Test-Time Augmentation with dropout")

🚀 Device: cuda

📂 Loading dataset...
✅ Training data: 3448 examples
✅ Labels: ['Ambivalent', 'Clear Non-Reply', 'Clear Reply']
✅ Class distribution: {'Clear Reply': 1052, 'Ambivalent': 2040, 'Clear Non-Reply': 356}

🔤 Setting up tokenizer...


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


🔄 Tokenizing training data...
✅ Tokenized 3448 examples

🎯 TRAINING 5-MODEL ENSEMBLE

[1/5] Training model 1...

🏋️  Training model with seed 42


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
50,1.063900
100,0.919100
150,0.806800
200,0.721800
250,0.616300
300,0.528600


...

Step,Training Loss
50,1.058800
100,0.898100
150,0.807500
200,0.748600
250,0.631300
300,0.523000


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


✅ Model saved to ./final_model_5

✅ All 5 models trained successfully!

📊 LOADING EVALUATION DATA


Map:   0%|          | 0/237 [00:00<?, ? examples/s]

✅ Evaluation data: 237 examples

🔮 GENERATING PREDICTIONS WITH TTA

🔮 Running 5-model ensemble with TTA...
   [1/5] Model 1 (with TTA)...
      ✅ Shape: torch.Size([237, 3])
   [2/5] Model 2 (with TTA)...
      ✅ Shape: torch.Size([237, 3])
   [3/5] Model 3 (with TTA)...
      ✅ Shape: torch.Size([237, 3])
   [4/5] Model 4 (with TTA)...
      ✅ Shape: torch.Size([237, 3])
   [5/5] Model 5 (with TTA)...
      ✅ Shape: torch.Size([237, 3])

🔄 Averaging ensemble predictions...
✅ Generated 237 predictions

📊 PREDICTION ANALYSIS

📊 Final prediction distribution:
   Ambivalent          :  104 ( 43.9%)
   Clear Non-Reply     :   36 ( 15.2%)
   Clear Reply         :   97 ( 40.9%)

💾 SAVING SUBMISSION
✅ Saved predictions to 'prediction' file
✅ Created clarity_final_submission.zip


📁 File: clarity_final_submission.zip
   ✓ 5 models
   ✓ Test-Time Augmentation with dropout
